# 环节 07 · GRPO 与 RLVR（配套 Notebook）

> 配套长文：[环节07-GRPO与RLVR详解.md](./环节07-GRPO与RLVR详解.md)
> 定位：组内归一化优势、std=0 退化、token 级损失、Clip-Higher、Dynamic Sampling。全部**纯 Python 标准库**。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 组内优势 | §3 / §10.1 | 用"同题采样一组"的均值替掉 critic |
| §2 std=0 | §3 | 全对/全错组没有梯度信号 |
| §3 token 级损失 | §5 / §10.2 | 长度差异大时序列级稀释 30× |
| §4 Clip-Higher | §5 / §10.3 | 抬高上界给低概率 token 留活路 |
| §5 Dynamic Sampling | §5 | 丢掉零梯度批次，代价是重采 |


## 1. GRPO 的核心：用"组内均值"当基线

同一个 prompt 采样 `G` 条回答，拿到 `G` 个奖励 → 组内标准化就是优势：

```
mean = Σ r_i / G
std  = sqrt(mean·(1 − mean))                    # 奖励是 0/1 时
A_i  = (r_i − mean) / std
```

**不需要 critic**（价值网络）—— 基线来自"这一组自己"，而不是另一个模型。


In [ ]:
import math

G = 8
print(f"{'k（对的条数）':>12} {'mean':>7} {'std':>7} {'A(对)':>9} {'A(错)':>9}")
for k in range(G + 1):
    mean = k / G
    std = math.sqrt(mean * (1 - mean))
    if std < 1e-12:
        print(f"{k:>12} {mean:>7.3f} {std:>7.3f}     ——        ——    ← 全同，无梯度")
    else:
        print(f"{k:>12} {mean:>7.3f} {std:>7.3f} {(1 - mean) / std:>+9.3f} {(0 - mean) / std:>+9.3f}")
print("→ k=1 时优势被放大到 +2.646（组里只有 1 条对）；k=4 时是最「标准」的 ±1。")


## 2. std = 0：GRPO 的头号故障

`k = 0`（全错）或 `k = G`（全对）时 `std = 0`：优势要么**爆炸**（除以 0）、要么**归零**。两种后果都很糟：训练悄悄不动，回报曲线是平的。


In [ ]:
eps = 1e-4
print("两种处理方式（k=0 全错 / k=8 全对，两种极端）：")
for k in (0, 8):
    mean = k / G
    std = math.sqrt(mean * (1 - mean))
    if k == 0:
        adv = (0 - mean) / max(std, eps)
        print(f"  k=0（全错）：std=0 → A(错) = {adv:+.0f} → 组内优势全为 0，无梯度")
    else:
        adv = (1 - mean) / max(std, eps)
        print(f"  k=8（全对）：std=0 → A(对) = {adv:+.0f} → 组内优势全为 0，无梯度")
print("  ① 加 ε 兜底：优势全部变 0（组内没有差异可比）；不加 ε 则是 0/0 → NaN 或爆炸")
print("  ② Dynamic Sampling：整组丢弃，不产生梯度（推荐做法）")


## 3. 序列级 vs token 级损失：长 CoT 必须用 token 级

- **序列级**：先对每条序列的 token 平均，再对序列平均 → 长回答里每个 token 的权重被"除以长度"稀释；
- **token 级**：所有 token 一起平均 → 权重与长度无关。


In [ ]:
for lens in ([50, 200, 800, 1500], [400, 420, 380, 440]):
    tot = sum(lens)
    seq = [1 / len(lens) / L for L in lens]           # 序列级：先平均序列、再除长度
    tok = [1 / tot] * len(lens)                       # token 级：全部 token 一起平均
    print(f"长度 = {lens}")
    print(f"  序列级权重 = {[f'{s:.2e}' for s in seq]}   最短/最长 = {seq[0] / seq[-1]:.1f}×")
    print(f"  token 级权重 = {[f'{t:.2e}' for t in tok]}   最短/最长 = {tok[0] / tok[-1]:.1f}×")
print("→ 长度差异大时序列级差 30.0×；token 级恒为 1.0×。这就是长 CoT 训不动的根因。")


## 4. Clip-Higher：给低概率 token 留成长空间

`ε` 决定"允许的**绝对**概率提升额度" = `ε · π_old`。低概率 token 的额度天然极小（比如 `π_old = 0.0071`、ε = 0.2 → 最多抬到 0.0085）→ 熵必然往下掉 → 策略"一根筋"。

**DAPO 的方案**：把上界抬到 `ε_high = 0.28`，让冷门 token 也能长。


In [ ]:
dpi = 0.002                                       # 想抬高的绝对概率
print(f"{'π_old':>8} {'ρ = 1 + Δπ/π_old':>18} {'ε=0.20':>16} {'ε=0.28':>16}")
for p_old in (0.001, 0.005, 0.0080, 0.01, 0.05):
    rho = 1 + dpi / p_old
    s20 = "截断(梯度 0)" if rho > 1.20 else "未截断"
    s28 = "截断(梯度 0)" if rho > 1.28 else "未截断"
    print(f"{p_old:>8} {rho:>18.3f} {s20:>16} {s28:>16}")
print("→ π_old=0.0080 时 ρ=1.250：ε=0.20 被截断、ε=0.28 仍有梯度 → 这就是 Clip-Higher 的用处。")


## 5. Dynamic Sampling：丢掉"零梯度批次"

采样时若某组全对或全错，它贡献不了梯度。**与其拿它去更新（除零/归零），不如整组丢弃、重新采样**。代价是额外算力，收益是每批都真的在学。


In [ ]:
import random

random.seed(0)
G, N, p = 8, 200, 0.35                              # 每组 8 条、共 200 组、单条正确率 0.35
keep = 0
for _ in range(N):
    k = sum(1 for _ in range(G) if random.random() < p)
    if 0 < k < G:
        keep += 1
print(f"单条正确率 p = {p}，每组 {G} 条，共 {N} 组")
print(f"可产生梯度的组（0 < k < G）：{keep}/{N} = {keep / N:.1%}")
print(f"被丢弃（全对 / 全错）：{N - keep} 组 = {(N - keep) / N:.1%}")
print("→ p 越极端（题太易/太难），丢弃比例越高：这是「难度要适中」的量化依据。")


## 6. 小结与下钻

- **GRPO = PPO − critic + 组基线**：用同题多次采样的统计量替代价值网络。
- **std=0 是结构性故障**：加 ε 或 Dynamic Sampling（推荐后者）。
- **长 CoT 必须 token 级损失**：序列级会稀释长回答的梯度（30×）。
- **Clip-Higher 抬高上界**给低概率 token 留活路，缓解熵坍缩。
- **奖励换验证器（RLVR）**：数学/代码用规则判分，摆脱 RM 的伪特征。

下一站：[环节 08 · Agentic RL 与信用分配](./环节08-AgenticRL与信用分配详解.md)（多步、多工具、多智能体）。
